# Treino do classificador

Fine-tuning de um ResNet-50 pré-treinado (ImageNet) sobre os recortes extraídos em `01_setup.ipynb`. A configuração padrão de `src/classification/train.py` reproduz a rota (b) YOLOv8 + ResNet-50 de Rosas-Alatriste et al. (*Scientific Reports*, 2026):

- Adam lr=1e-3, batch 32, até 100 épocas, seed 0.
- Primeiras ~50 camadas congeladas (`conv1`/`bn1`/`layer1`/`layer2[0]`), com o BatchNorm em modo inferência, como no Keras.
- Cross-entropy sem peso de classe nem label smoothing.
- `ReduceLROnPlateau(factor=0.02, patience=3, min_lr=1e-6)`, `EarlyStopping(patience=15)` e checkpoint pelo menor `val_loss`.
- Imagem esticada para 224x224 e augmentation do `ImageDataGenerator` (rotação ≤30°, zoom ≤20%, shift ≤10%, flip horizontal).

Diferenças em relação ao paper: o treino usa os recortes do próprio dataset de metáfases (o paper usa o Dataset B, de cromossomos Q-band isolados), e o split 70/15/15 é o mesmo do detector, feito por metáfase -- todos os cromossomos de uma metáfase ficam no mesmo split.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src" / "classification"))
sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))

from train import DEFAULT_CONFIG, train

RUN_NAME = "resnet50-paper"
RESUME = True  # continua de last.pt se a run já existir (sem efeito numa run nova)
NUM_WORKERS = 8

# Sobrescreve campos da configuração padrão. Ex.: {"resize": "letterbox"} para a ablação com
# letterbox (o resize esticado do paper distorce recortes muito alongados) -- use
# outro RUN_NAME, já que o resume recusa config diferente da run salva.
OVERRIDES = {}

{**DEFAULT_CONFIG, **OVERRIDES}

Roda o treino. Com early stopping, ele para bem antes das 100 épocas. Salva em `models/classifier/runs/<RUN_NAME>/`: `best.pt` (menor `val_loss`), `last.pt` (estado completo para retomar, inclusive dos geradores aleatórios -- fora do git), `history.csv` (loss, acurácia e macro-F1 por época) e `config.json`.

Com `RESUME = True`, o treino continua da última época salva. O mesmo treino pode ser executado pelo terminal:

```bash
uv run python src/classification/train.py --run-name resnet50-paper --resume
```

In [ ]:
best_checkpoint = train(
    run_name=RUN_NAME, resume=RESUME, num_workers=NUM_WORKERS, **OVERRIDES
)
print(f"Checkpoint: {best_checkpoint}")

Curvas de treino/validação salvas em `history.csv`. Como no paper, a convergência e o overfitting são avaliados pelas curvas de loss e acurácia (a acurácia de validação é a métrica comparável aos 94.04% reportados pelo paper); a learning rate mostra quando o `ReduceLROnPlateau` atuou.

In [ ]:
import json

import pandas as pd
import matplotlib.pyplot as plt

run_dir = best_checkpoint.parent
history = pd.read_csv(run_dir / "history.csv")
monitor = json.loads((run_dir / "config.json").read_text())["monitor"]
best_row = history[monitor].idxmin() if monitor == "val_loss" else history[monitor].idxmax()
best_epoch = history.loc[best_row, "epoch"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
panels = [(axes[0, 0], "loss", "Loss"), (axes[0, 1], "acc", "Acurácia"), (axes[1, 0], "f1", "Macro-F1")]
for ax, metric, title in panels:
    ax.plot(history["epoch"], history[f"train_{metric}"], label="train")
    ax.plot(history["epoch"], history[f"val_{metric}"], label="val")
    ax.axvline(best_epoch, color="gray", linestyle="--", label=f"best (época {best_epoch})")
    ax.set_title(title)
    ax.set_xlabel("época")
    ax.legend()

axes[1, 1].plot(history["epoch"], history["lr"])
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("Learning rate")
axes[1, 1].set_xlabel("época")
plt.tight_layout()
plt.show()